# 06 — Assembly Policies & Knowledge

Research agents run long conversations with many tool calls. Without context management, they drown in their own history. AG2 beta provides two mechanisms:

1. **Assembly policies** — filter and window events before each model call.
2. **KnowledgeConfig** — persistent storage with automatic compaction.

lionag2 configures both on every agent.

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()

from autogen.beta import Agent, KnowledgeConfig, MemoryStream
from autogen.beta.compact import CompactTrigger, TailWindowCompact
from autogen.beta.config import OpenAIConfig
from autogen.beta.knowledge import MemoryKnowledgeStore
from autogen.beta.policies import ConversationPolicy, SlidingWindowPolicy

config = OpenAIConfig("gpt-5.4-mini", api_key=os.getenv("OPENAI_API_KEY"))

## Assembly policies

lionag2 stacks two policies on every agent:

- `ConversationPolicy()` — keeps only conversation + tool events (filters out internal bookkeeping).
- `SlidingWindowPolicy(max_events=40, transparent=True)` — keeps the last 40 events. `transparent=True` means the agent doesn't know events were dropped.

Together they give bounded, relevant context without manual truncation.

In [ ]:
agent = Agent(
    "demo",
    prompt="You are a helpful research assistant.",
    config=config,
    assembly=[
        ConversationPolicy(),
        SlidingWindowPolicy(max_events=40, transparent=True),
    ],
)

reply = await agent.ask("How does context management work in multi-agent research?")
print(reply.body[:500])

## KnowledgeConfig

The knowledge store persists across agent turns. AG2's `WorkingMemoryPolicy` and `EpisodicMemoryPolicy` read/write through it automatically.

lionag2 configures:
- `store` — where knowledge lives (in-memory or khive-backed).
- `compact` — `TailWindowCompact(target=30)` keeps the last 30 events when compacting.
- `compact_trigger` — `CompactTrigger(max_events=50)` triggers compaction after 50 events.

In [ ]:
store = MemoryKnowledgeStore()

knowledge = KnowledgeConfig(
    store=store,
    compact=TailWindowCompact(target=30),
    compact_trigger=CompactTrigger(max_events=50),
)

agent_with_knowledge = Agent(
    "researcher",
    prompt="You are a researcher with persistent memory.",
    config=config,
    knowledge=knowledge,
    assembly=[
        ConversationPolicy(),
        SlidingWindowPolicy(max_events=40, transparent=True),
    ],
)

print(f"Store contents: {await store.list('/')}")

reply = await agent_with_knowledge.ask("What do we know about chain-of-thought prompting?")
print(f"Reply: {reply.body[:300]}")
print(f"Store after run: {await store.list('/')}")

## How the engine configures agents

Every agent in `ResearchEngine._make_agent()` gets the same assembly config:

```python
agent = Agent(
    spec["name"],
    prompt=spec["prompt"],
    config=self.config,
    tools=tools,
    assembly=[
        ConversationPolicy(),
        SafeSlidingWindowPolicy(max_events=40, transparent=True),
    ],
)
```

`SafeSlidingWindowPolicy` extends AG2's `SlidingWindowPolicy` with full tool-call/result pairing integrity — orphaned tool results are filtered at the individual result level.

## Up next

Tutorial 07 introduces the 6 specialist agents that make up a research team — each with specific tools, depth-aware prompts, and a defined handoff order.